In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, to_date, when

bronze_df = spark.table(
    "workspace.telecom_bronze.network_events"
)

bronze_df.count()
valid_silver_df = (
    bronze_df
    .dropDuplicates(["event_id"])
    .filter(col("signal_strength").between(-130, -40))
    .filter(col("latency_ms") > 0)
    .withColumn(
        "network_quality",
        when(col("latency_ms") <= 100, "EXCELLENT")
        .when(col("latency_ms") <= 250, "GOOD")
        .otherwise("POOR")
    )
    .withColumn("event_date", to_date(col("event_timestamp")))
)
rejected_df = (
    bronze_df
    .filter(
        (~col("signal_strength").between(-130, -40)) |
        (col("latency_ms") <= 0) |
        col("event_id").isNull() |
        col("event_timestamp").isNull()
    )
)
valid_silver_df.write.mode("overwrite").saveAsTable("workspace.telecom_silver.network_events")
rejected_df.write.mode("overwrite").saveAsTable("workspace.telecom_silver.network_events_rejected")


In [0]:
%sql

SELECT
    network_quality,
    COUNT(*) cnt
FROM workspace.telecom_silver.network_events
GROUP BY network_quality
ORDER BY cnt DESC;

network_quality,cnt
POOR,25435
GOOD,15334
EXCELLENT,9231


In [0]:
silver_df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- cell_tower_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- network_type: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- signal_strength: integer (nullable = true)
 |-- latency_ms: integer (nullable = true)
 |-- dropped_call: integer (nullable = true)
 |-- data_usage_mb: double (nullable = true)
 |-- network_quality: string (nullable = false)
 |-- event_date: date (nullable = true)



In [0]:
bronze_df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- cell_tower_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- network_type: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- signal_strength: integer (nullable = true)
 |-- latency_ms: integer (nullable = true)
 |-- dropped_call: integer (nullable = true)
 |-- data_usage_mb: double (nullable = true)

